# Amazon SageMaker - Tổng quan & Kiến trúc Training & Deployment

## 1. SageMaker là gì?
- **Amazon SageMaker**: Dịch vụ **fully-managed** chính của AWS dành cho **Machine Learning**
- Hỗ trợ **toàn bộ ML lifecycle** (end-to-end):
  - Thu thập & chuẩn bị dữ liệu
  - Training & evaluation model
  - Hyperparameter tuning
  - Deploy model vào production
  - Monitoring & inference
- Hỗ trợ **traditional ML**, **Deep Learning**, và cả **Generative AI** (nhưng GenAI thuần dùng Bedrock tốt hơn)
- SageMaker tồn tại từ trước khi GenAI bùng nổ → tập trung rộng vào ML engineering

> **Quan trọng cho thi MLA-C01**: SageMaker là service **core**, chiếm tỷ lệ lớn ở Domain 2 & Domain 3.

## 2. Kiến trúc Training & Deployment (Conceptual Flow)

### Bottom-up (Quy trình thực tế):

**Training Phase:**
- **Training Code** (container image) → lưu trong **Amazon ECR**
- **Training Data** → lấy từ **Amazon S3** (đã chuẩn bị sẵn)
- Training Job chạy → sinh ra **Model Artifacts** → lưu vào **S3**

**Deployment / Inference Phase:**
- Model Artifacts (S3) + **Inference Code** (container từ ECR)
- Deploy thành **SageMaker Endpoint(s)** (có thể scale nhiều)
- **Client Application** → gọi inference qua **Endpoint**

### Sơ đồ tóm tắt:



**Training Job → Output:**
- Input: Training Image (ECR) + Data (S3)
- Output: Model Artifacts (S3)

## 3. Các cách sử dụng SageMaker

| Cách thực hiện              | Mô tả                                                                 | Phù hợp với |
|-----------------------------|-----------------------------------------------------------------------|-------------|
| **SageMaker Notebook**      | Jupyter Notebook (chạy trên EC2), viết Python code. Có sẵn thư viện (scikit-learn, TensorFlow, PyTorch, Spark...) | Data Scientist, ML Engineer thích code |
| **SageMaker Console (UI)**  | Làm hoàn toàn qua giao diện web, không cần code nhiều. Dùng built-in algorithms | Người mới, nhanh prototype |
| **SageMaker Python SDK**    | Viết code chuyên nghiệp (Estimator, Processor, Pipeline)              | Production, MLOps, automation |

**Đặc điểm Notebook:**
- Tự động spin up/down trên EC2
- Có sẵn kết nối S3
- Nhiều thư viện ML phổ biến được cài sẵn
- Có thể orchestrate toàn bộ: Data Processing → Training → Tuning → Deploy

## 4. Điểm nhấn thi MLA-C01
- SageMaker **không chỉ là nơi train model** mà là **platform quản lý toàn bộ ML workflow**
- Hiểu rõ vai trò:
  - **S3**: Lưu training data & model artifacts
  - **ECR**: Lưu training & inference container images
  - **Endpoint**: Nơi phục vụ prediction ở production (real-time / batch)
- Có thể dùng **built-in algorithms** mà không cần viết code thuật toán từ đầu

**Mẹo note**: 
- Tập trung vào **"Training Job → Model Artifacts → Endpoint"**
- Hiểu cách orchestrate từ Notebook hoặc Console

---

# SageMaker Domain - Kiến trúc & Cấu hình Cơ bản

## 1. SageMaker Domain là gì?
- **SageMaker Domain** là **tổ chức chính (umbrella)** để sử dụng SageMaker Studio.
- **Bắt buộc phải tạo Domain** trước khi dùng bất kỳ tính năng nào của SageMaker.
- Toàn bộ hoạt động trong SageMaker (users, notebooks, apps, data…) đều nằm trong **một Domain**.

> **Quan trọng thi MLA-C01**: Domain là điểm khởi đầu của mọi kiến trúc SageMaker.

## 2. Thành phần chính trong SageMaker Domain

- **EFS Volume** (Elastic File System):
  - Một EFS volume duy nhất được tạo dưới domain.
  - Được chia sẻ cho toàn domain (shared + private directories).

- **User Profiles**:
  - Đại diện cho từng người dùng (user-centric).
  - Mỗi user profile có:
    - Private EFS directory (không gian riêng tư).
    - Personal applications (SageMaker Studio, Canvas…).

- **Shared Resources**:
  - Shared EFS directory (dùng chung giữa các user).
  - Shared Spaces: nơi chia sẻ notebooks, data, kết quả giữa nhiều người.

- **Applications**:
  - SageMaker Studio (IDE chung).
  - Các ứng dụng cá nhân của từng user profile.

## 3. VPC & Network Configuration (Phần quan trọng)

Khi tạo SageMaker Domain, mặc định có **2 VPC**:

| VPC Type                  | Mục đích                                                                 | Quản lý bởi     |
|---------------------------|--------------------------------------------------------------------------|-----------------|
| **Internet-facing VPC**   | Kết nối ra internet (tải package, data từ public, deploy ra ngoài)      | SageMaker quản lý tự động |
| **Customer VPC**          | Truy cập EFS volume (dữ liệu private), traffic nội bộ                    | Bạn phải chỉ định |

**Các thông tin cần cấu hình khi tạo Domain:**
- VPC ID (của bạn)
- Subnets (thường chọn tất cả subnets)
- Security Groups
- VPC-only mode (không dùng internet VPC của SageMaker)

### Hai chế độ VPC phổ biến:
1. **Default mode** (2 VPC): Dễ dùng, có internet access.
2. **VPC-only mode**: Tất cả traffic đi qua VPC của bạn → bảo mật cao hơn, kiểm soát chặt chẽ (khuyến nghị production).

## 4. Tóm tắt SageMaker Domain
- Là **organizational unit** lớn nhất trong SageMaker.
- Bao gồm: Users, User Profiles, Shared Spaces, Applications, EFS, VPC settings.
- Mỗi user có không gian riêng + có thể chia sẻ tài nguyên.
- Network security được quản lý chủ yếu qua **VPC + Subnets + Security Groups**.

**Mẹo ôn thi**:
- Phải tạo Domain trước khi dùng SageMaker Studio.
- Hiểu rõ sự khác biệt giữa **Private directory** và **Shared directory**.
- VPC-only mode thường được dùng trong môi trường production/secure.

---

# SageMaker - Data Preparation → Training → Deployment

## 1. Data Preparation (Data Prep)

**Nguồn dữ liệu chính:**
- **Amazon S3** (phổ biến nhất)
- **FSx for Lustre** (dành cho workload quy mô lớn, hiệu suất cao)
- Các nguồn khác:
  - Amazon Athena
  - Elastic MapReduce (EMR)
  - Amazon Redshift
  - Amazon Keyspaces

**Công cụ xử lý dữ liệu:**
- Tích hợp **Apache Spark** (không chỉ AWS)
- Python libraries có sẵn trong SageMaker Notebook:
  - scikit-learn, NumPy, pandas
- **Processing Container**:
  - Dùng **built-in processing containers** (không cần viết code)
  - Hoặc custom container (từ ECR)

**Quy trình đơn giản:**
1. Copy data từ S3 vào Processing Job
2. Xử lý → Output ra **S3 bucket mới** (đã cleaned & formatted)
3. Dữ liệu sau xử lý sẽ được truyền sang Training Job

> **Lưu ý**: Định dạng dữ liệu nên phù hợp với algorithm (RecordIO, protobuf, columnar…).

## 2. Training Model

**Tạo Training Job:**
- Input: Đường dẫn S3 chứa dữ liệu đã xử lý
- Chọn **compute resources** (instance type & số lượng) → chi phí cao ở bước này
- Training code: Lưu trong **Amazon ECR** (container)

**Các lựa chọn Training Code:**
- **Built-in Algorithms** (SageMaker cung cấp sẵn)
- **Framework Containers**:
  - TensorFlow, PyTorch, MXNet
  - Spark ML
  - scikit-learn
  - Reinforcement Learning (RL)
  - XGBoost
  - Hugging Face (LLM, SLM, GenAI models)
  - Chainer
- **Custom Docker Image** (bất kỳ thứ gì bạn muốn)
- Algorithms mua từ **AWS Marketplace**

**Output của Training:**
- **Model Artifacts** → lưu vào **S3 bucket** chỉ định

## 3. Deployment (Triển khai Model)

**2 cách triển khai chính:**

| Loại                  | Tên tính năng                  | Mô tả                                                                 | Khi nào dùng?                  |
|-----------------------|--------------------------------|-----------------------------------------------------------------------|--------------------------------|
| **Real-time**         | SageMaker Endpoint             | Persistent endpoint, scale tự động, phục vụ prediction ngay lập tức   | Cần prediction real-time       |
| **Batch**             | SageMaker Batch Transform      | Xử lý batch lớn một lần, không cần endpoint                           | Dự đoán hàng loạt, không realtime |

**Tính năng nâng cao khác:**
- **Inference Pipelines**: Orchestrate chuỗi xử lý phức tạp trước/sau inference
- **SageMaker Neo**: Deploy model ra **edge devices** (thiết bị biên, ít kết nối internet)
- **Elastic Inference**: Tăng tốc inference (gắn thêm accelerator)
- **Automatic Scaling**: Tự động scale số endpoint theo traffic
- **Shadow Testing**: Test model mới song song với model cũ (không ảnh hưởng production)
- **Rollback**: Dễ dàng rollback nếu model mới không đạt yêu cầu

**Mẹo thi MLA-C01**:
- Hiểu rõ flow: **S3 (raw) → Processing Job → S3 (processed) → Training Job → Model Artifacts (S3) → Endpoint / Batch Transform**
- SageMaker hỗ trợ **rất nhiều framework** (built-in + custom container)
- Deployment không chỉ có real-time mà còn batch, edge, auto-scale, shadow test

---

# SageMaker Ground Truth - Dịch vụ Gán nhãn dữ liệu

## 1. SageMaker Ground Truth là gì?
- Dịch vụ của AWS giúp **sử dụng con người để gán nhãn (label) dữ liệu** cho mục đích huấn luyện ML.
- Thường dùng khi dữ liệu thiếu **labels** (nhãn) hoặc thiếu **features**.
- Phổ biến nhất trong **Computer Vision** (ví dụ: gán nhãn ảnh là bóng rổ hay bóng đá, tìm chim trong ảnh…).

> **Vị trí trong Feature Engineering**: Vì nó giúp tạo/gán nhãn và tạo thêm features cho mô hình.

## 2. Cách Ground Truth hoạt động (Thông minh & Tiết kiệm)

- Ban đầu: Gửi task cho **con người** để gán nhãn.
- Trong quá trình: Ground Truth **tự động xây dựng một model** dựa trên nhãn từ con người.
- Sau đó: Chỉ gửi những case **khó / mơ hồ** cho con người. Những case dễ thì model tự dự đoán.
- **Lợi ích**: Giảm chi phí labeling lên đến **70%**.

## 3. Các loại Workforce (Đội ngũ gán nhãn)

| Loại Workforce                  | Mô tả                                                                 | Phù hợp khi |
|--------------------------------|-----------------------------------------------------------------------|-------------|
| **Amazon Mechanical Turk**     | Đội ngũ lớn trên toàn thế giới, chi phí thấp                          | Dự án thông thường, ngân sách hạn chế |
| **Private Workforce**          | Đội ngũ nội bộ của công ty bạn                                       | Dữ liệu nhạy cảm, bảo mật cao |
| **Vendor Workforce**           | Công ty chuyên labeling chuyên nghiệp                                 | Cần chất lượng cao, chuyên sâu |

## 4. Ground Truth Plus (Dịch vụ cao cấp)
- AWS làm **toàn bộ** thay bạn (turnkey solution).
- Đội ngũ chuyên gia AWS thiết lập workflow, quản lý labeler, theo dõi tiến độ.
- Bạn chỉ cần điền form mô tả yêu cầu → AWS liên hệ báo giá (không công khai, thường đắt).
- Theo dõi tiến độ qua **Ground Truth Plus Project Portal** (có dashboard, charts).
- Kết quả cuối cùng: Nhận dữ liệu đã gán nhãn từ **S3**.

## 5. Các cách thay thế / Bổ sung (Không dùng người)

- **Amazon Rekognition**: Dùng pre-trained model để tự động gán nhãn ảnh/video (object detection, classification…).
- **Amazon Comprehend**: Phân tích text → tạo features như topic, sentiment, entity… (rất hữu ích cho NLP).
- Bất kỳ **pre-trained model** hoặc **unsupervised learning** nào cũng có thể dùng để sinh thêm labels hoặc features.

> **Tóm tắt**: Ground Truth dùng để tạo **ground truth labels** khi máy không làm được. Kết hợp với Rekognition/Comprehend để giảm chi phí và tăng tốc độ feature engineering.

## 6. Điểm nhấn thi MLA-C01
- Ground Truth thuộc phần **Data Preparation / Feature Engineering**.
- Hiểu rõ cơ chế **Active Learning** (model tự học và chỉ gửi case khó cho người).
- Biết sự khác biệt giữa Ground Truth thông thường và **Ground Truth Plus**.
- Có thể kết hợp với AI services (Rekognition, Comprehend) để tự động hóa labeling.

---

# Amazon Mechanical Turk (MTurk)

## 1. Mechanical Turk là gì?
- Dịch vụ **crowdsourcing marketplace** của AWS.
- Cho phép bạn tiếp cận **đội ngũ lao động ảo phân tán** (virtual workforce) trên toàn thế giới.
- Con người sẽ thực hiện các **task đơn giản** với chi phí rất rẻ.

**Nguồn gốc tên gọi:**
- Lấy cảm hứng từ “Mechanical Turk” năm 1770: Một robot chơi cờ vua giả tạo (thực ra có người ẩn bên trong điều khiển).

## 2. Cách hoạt động
- Bạn tạo **tasks** (công việc) trên Mechanical Turk.
- Con người khắp nơi trên thế giới nhận làm và hoàn thành task.
- Bạn **quyết định giá thưởng** (reward) cho mỗi task.
- Ví dụ: 
  - 10 triệu ảnh, reward 0.1 USD/ảnh → tổng chi phí 1 triệu USD.

**Giao diện cho Worker:**
- Worker thấy danh sách jobs + reward.
- Họ chấp nhận job → làm → nhận tiền.

## 3. Use Cases phổ biến
- **Image classification / labeling** (gán nhãn ảnh)
- Data collection
- Data validation / review
- Business process outsourcing (nhập liệu, điền form, kiểm tra Excel…)
- Review recommendations

## 4. Vai trò trong Machine Learning & AI
- Dùng để **tạo nhãn dữ liệu (labeling)** cho tập huấn luyện.
- Hỗ trợ **feature engineering** khi cần dữ liệu do con người tạo ra.
- Tích hợp sâu với:
  - **SageMaker Ground Truth**
  - **Amazon A2I** (Augmented AI)

## 5. Điểm nhấn thi MLA-C01
- Mechanical Turk là một loại **Workforce** trong SageMaker Ground Truth.
- Phù hợp khi cần **lượng dữ liệu lớn** với chi phí thấp.
- Ưu điểm: Nhanh, rẻ, quy mô lớn.
- Nhược điểm: Chất lượng phụ thuộc vào reward và task design; không phù hợp với dữ liệu nhạy cảm.

**So sánh nhanh với Ground Truth:**
- Mechanical Turk = Workforce công khai, chi phí thấp.
- Private Workforce = Đội ngũ nội bộ (bảo mật cao hơn).
- Vendor Workforce = Đội chuyên nghiệp.

---